# 07 — 差异表达分析 (Differential Expression)

本 notebook 做两类差异表达分析：

1. **Per-cluster DEG**：每个 Leiden 簇的标记基因（用于簇注释与功能解读）
2. **跨疾病阶段 DEG**：比较 CAG→IM→Dysplasia 轴上不同疾病阶段的差异基因

方法：`sc.tl.rank_genes_groups`（Wilcoxon 秩和检验），scanpy 原生实现。
产出：per-cluster 标记基因表 + 疾病轴 DEG 表 → 导出 CSV + 写入 `adata.uns`。

**为什么不做复杂模型**（MAST、DESeq2 per-cell）？单细胞 DEG 的 p-value 在
数万细胞尺度上过于 optimistic（pseudoreplication 问题），per-cell 模型会把
采样差异误判为生物学差异。本 notebook 的 per-cluster DEG 用于定性的簇注释
（"这个簇高表达哪些基因"），不做严格的统计推断。要做可靠的统计比较，
参照 `pseudobulk_deg.ipynb`——在 sample 层面聚合后用 DESeq2。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：06（注释），读 `06_annotated_v*.h5ad`
- **下游**：下游分析（通路富集 / GRN / 火山图展示），产出 `07_deg_v*.h5ad` + DEG 表格

### 为什么要迭代回跑？
差异表达分析 (deg) 的结果是下游分析和 PI 生物学判断的基础。如果在后续分析中发现：
- DEG 阈值过高导致遗漏关键基因、过低导致假阳性
- 通路富集缺少预期应出现的生物学通路
- 调控网络缺少已知的主控转录因子
- CNV 信号不符合病理学预期
可能需要调整本 stage 的参数重新计算。

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
2. **改 `OUTPUT_PATH`**——bump 版本号 `_v1` → `_v2`（旧版不覆盖）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改以下参数后重跑本 notebook）：
- `LEIDEN_COL` — 切换簇分组列
- `N_TOP_GENES_PER_CLUSTER` — 每簇标记基因数
- `LOG2FC_CUTOFF` / `PVAL_ADJ_CUTOFF` — 差异显著性阈值
- `DISEASE_CONTRASTS` — 调整疾病对比对

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为结果可接受、可传给下游使用的正式版本。
  PI 在 Jupyter 中打开 `.h5ad` 后手动改 `adata.uns["status"] = "promoted"` 再保存。
- **下游取数**：下游 notebook 的 `UPSTREAM_PATH` 指向你决定采用的版本即可。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"07_deg"`（本 stage 标识）
- `status` = `"experimental"`（PI 审查后改为 `"promoted"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_PATH` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询「差异表达分析 (deg) 有哪些版本？哪些依赖 06_annotated_v1？」，
可直接在 Python 中 glob `results/` 目录检查每个 `.h5ad` 的 `adata.uns`。

In [ ]:
# === PARAMS ===
# UPSTREAM_PATH         — 06 注释结果 h5ad（含 cell_type_final_v1）
# OUTPUT_PATH         — 本 stage 产出 checkpoint 路径。
#                        版本号 _v1 与 adata.uns['version'] 保持一致。
#                        如需回跑：修改上方 OUTPUT_VERSION 即可，旧版不覆盖。
# LEIDEN_COL            — Leiden 簇标签 obs 列
# DISEASE_COL           — 疾病阶段 obs 列
#                         **重要提醒**：此列需在 01/02 已填充。
#                         多数真实数据集的 obs 中不包含此列；若列为空或不存在，
#                         请先在 02 的 obs 中填好疾病分期列并修改本 PARAMS。
#                         常见实际列名：disease_stage、condition、group 等。
# DISEASE_CONTRASTS     — 要做 DEG 的对比，[(numerator, denominator), ...]
# MIN_CELLS_PER_GROUP   — 每组最少细胞数才能做 DEG
# N_TOP_GENES_PER_CLUSTER — 每簇取 top 标记基因数
# LOG2FC_CUTOFF         — logFC 绝对值阈值（用于筛选显著 DEG）
# PVAL_ADJ_CUTOFF       — 调整后 p 值阈值

# === 版本变量（单点定义，回跑时只改这里，下游全部引用）===
# 为什么用字符串而非整数：uns["version"] 历来存 "v1"，直接引用变量即保持语义不变；
# 回跑时把 "v1" 改成 "v2" 即可，旧结果不覆盖。
UPSTREAM_VERSION = "v1"   # 上游输入 checkpoint 的版本号（本 stage 读取的 06/D04 产物）
OUTPUT_VERSION   = "v1"   # 本 stage 产出 checkpoint 的版本号

UPSTREAM_PATH = f"results/06_annotated_{UPSTREAM_VERSION}.h5ad"
OUTPUT_PATH   = f"results/07_deg_{OUTPUT_VERSION}.h5ad"

LEIDEN_COL = "leiden_res_0.6"
DISEASE_COL = "disease"  # best-effort 默认值——多数数据集可能为空，PI 必须核对实际 obs 列名
DISEASE_CONTRASTS = [
    ("CAG", "normal"),          # 萎缩 vs 正常
    ("IM", "CAG"),              # 肠化 vs 萎缩
    ("dysplasia", "IM"),        # 异型增生 vs 肠化
]
MIN_CELLS_PER_GROUP = 10
N_TOP_GENES_PER_CLUSTER = 50
LOG2FC_CUTOFF = 0.25    # ~1.2 倍变化
PVAL_ADJ_CUTOFF = 0.05

In [ ]:
# === setup：sys.path + 导入 + 加载上游 ===

# 1. 确保框架 src/ 在 sys.path 并切换到项目根目录
#    多级回退策略：nbconvert/conda run 的 CWD 不稳定，
#    先试 CWD，再试从 notebooks/07_downstream/ 回退两级
import sys, os, gc
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")
print(f"src 存在: {os.path.isdir(os.path.join(_root, 'src', 'scrna_integration'))}")
# 2. 导入依赖
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
sc.settings.figdir = "results/figures"
warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(42)

print(f"scanpy {sc.__version__}  |  numpy {np.__version__}")
# 3. 加载上游 06 输出
#    契约：需包含 cell_type_final_v1、LEIDEN_COL、
#    以及（若做跨疾病 DEG）DISEASE_COL
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"obs 列: {list(adata.obs.columns)}")
print(f"leiden 列 '{LEIDEN_COL}' 存在: {LEIDEN_COL in adata.obs.columns}")
if LEIDEN_COL in adata.obs.columns:
    print(f"  簇数: {adata.obs[LEIDEN_COL].nunique()}")
print(f"DISEASE_COL '{DISEASE_COL}' 存在: {DISEASE_COL in adata.obs.columns}")
if DISEASE_COL in adata.obs.columns:
    print(f"  疾病阶段: {sorted(adata.obs[DISEASE_COL].dropna().unique())}")
print(f"cell_type_final_v1 存在: {'cell_type_final_v1' in adata.obs.columns}")
if "cell_type_final_v1" in adata.obs.columns:
    print(f"  细胞类型: {sorted(adata.obs['cell_type_final_v1'].dropna().unique())}")

## Per-cluster DEG — 每簇标记基因

用 `sc.tl.rank_genes_groups` 对每个 Leiden 簇找标记基因。
**为什么用 Wilcoxon？** 它是非参数检验，不假设表达量服从正态分布；
scanpy 默认参数经过充分验证，对稀疏 scRNA-seq 数据鲁棒性好。
"1-vs-rest" 模式下，每个簇与所有其他簇的细胞比较，找出该簇特异高表达的基因。

结果用途：簇的功能注释、后续通路富集的输入。

In [ ]:
# Per-cluster 差异表达: 每个簇 vs 所有其他簇。
# 为什么 n_genes=50？足够覆盖生物学功能注释需要的标记基因数量，
# 又不至于因为过多低 fold-change 基因稀释下游富集分析信号。
if LEIDEN_COL in adata.obs.columns:
    # 确保 leiden 列是 category 或 str
    if not hasattr(adata.obs[LEIDEN_COL], "cat"):
        adata.obs[LEIDEN_COL] = adata.obs[LEIDEN_COL].astype(str)

    sc.tl.rank_genes_groups(
        adata,
        groupby=LEIDEN_COL,
        method="wilcoxon",
        n_genes=N_TOP_GENES_PER_CLUSTER,
        key_added="rank_genes_per_cluster",
        use_raw=False,
    )
    print(f"per-cluster DEG 完成 (key='rank_genes_per_cluster')")
else:
    print(f"LEIDEN_COL '{LEIDEN_COL}' 不存在，跳过 per-cluster DEG")

In [ ]:
# 提取每簇 top 标记基因 → DataFrame。
# 为什么收束为一个表？PI 可以一次浏览所有簇的标记基因，而不是逐个簇翻 adata.uns。
_per_cluster_records = []
if "rank_genes_per_cluster" in adata.uns:
    _cluster_ids = sorted(adata.obs[LEIDEN_COL].astype(str).unique())
    for _cid in _cluster_ids:
        try:
            _df = sc.get.rank_genes_groups_df(
                adata, group=_cid, key="rank_genes_per_cluster"
            )
            _df["cluster"] = _cid
            _per_cluster_records.append(_df.head(N_TOP_GENES_PER_CLUSTER))
        except Exception:
            # rank_genes_groups 可能因簇太小而没结果
            pass

    if _per_cluster_records:
        _per_cluster_all = pd.concat(_per_cluster_records, ignore_index=True)
        _per_cluster_csv = "results/tables/07_per_cluster_markers.csv"
        _per_cluster_all.to_csv(_per_cluster_csv, index=False)
        print(f"per-cluster 标记基因表 ({len(_per_cluster_all)} 条) 已保存: {_per_cluster_csv}")

        # 简要展示 top 3 per cluster
        _top3 = _per_cluster_all.groupby("cluster").head(3)
        print("\n每簇 top 3 标记基因（预览）:")
        print(_top3[["cluster", "names", "logfoldchanges", "pvals_adj"]].to_string(index=False))
    else:
        print("per-cluster DEG 无结果——可能所有簇都太小")
else:
    print("rank_genes_per_cluster 不在 adata.uns 中，跳过提取")

### Heatmap: per-cluster top 标记基因

每簇取 top 3 标记基因，画跨簇的 mean expression heatmap。
**为什么看跨簇表达模式？** 一个簇的"marker gene"应该在它标注的簇中
特异高表达，而不是广谱表达——热图能一眼确认 marker 的特异性。

In [ ]:
# Per-cluster top marker heatmap。
# 每簇取 top 3，去重后取各基因在各簇的 mean expression。
if "rank_genes_per_cluster" in adata.uns and LEIDEN_COL in adata.obs.columns:
    _top_genes = set()
    _cluster_ids = sorted(adata.obs[LEIDEN_COL].astype(str).unique())
    for _cid in _cluster_ids:
        try:
            _df = sc.get.rank_genes_groups_df(
                adata, group=_cid, key="rank_genes_per_cluster"
            )
            _top_genes.update(_df["names"].head(3).tolist())
        except Exception:
            pass

    _top_genes = sorted(_top_genes)
    if len(_top_genes) > 0 and len(_top_genes) <= 100:
        sc.pl.heatmap(
            adata,
            var_names=_top_genes,
            groupby=LEIDEN_COL,
            show_gene_labels=True,
            standard_scale="var",
            figsize=(12, max(4, len(_top_genes) * 0.25)),
            show=False,
            save="_07_per_cluster_heatmap.png",
        )
        plt.close("all")
else:
    print("无 per-cluster DEG 结果，跳过 heatmap")


## 跨疾病阶段 DEG — CAG → IM → Dysplasia 轴

沿着胃"炎-癌"转化轴（正常 → CAG → IM → Dysplasia），
比较相邻疾病阶段的差异表达基因。
**为什么相邻比较？** 线性轴的相邻对比最能捕捉疾病进展中每一步的具体
分子变化，比"终末 vs 正常"的混合信号更有生物学解释力。

疾病阶段值对应关系（PI 根据实际数据确认）：
- `normal`：正常胃粘膜
- `CAG`：慢性萎缩性胃炎
- `IM`：肠上皮化生
- `dysplasia`：异型增生

In [ ]:
# 跨疾病阶段 DEG——相邻阶段对比。
# 每一步对比：(numerator, denominator) = (更晚期, 更早期)
# logFC > 0 → 在更晚期高表达（随疾病进展上调）
# logFC < 0 → 在更早期高表达（随疾病进展下调）
_disease_deg_results = {}
_disease_col_available = DISEASE_COL in adata.obs.columns

if _disease_col_available:
    # 检查每个 contrast 两组的细胞数
    _disease_counts = adata.obs[DISEASE_COL].value_counts()
    print(f"疾病阶段细胞分布: {dict(_disease_counts)}")

    for _num, _den in DISEASE_CONTRASTS:
        _n_num = _disease_counts.get(_num, 0)
        _n_den = _disease_counts.get(_den, 0)

        if _n_num < MIN_CELLS_PER_GROUP or _n_den < MIN_CELLS_PER_GROUP:
            print(f"  跳过 {_num} vs {_den}: 细胞数不足 "
                  f"({_num}={_n_num}, {_den}={_n_den}, min={MIN_CELLS_PER_GROUP})")
            continue

        print(f"  做 DEG: {_num} (n={_n_num}) vs {_den} (n={_n_den})")

        _key = f"rank_genes_disease_{_num}_vs_{_den}"
        sc.tl.rank_genes_groups(
            adata,
            groupby=DISEASE_COL,
            groups=[_num],
            reference=_den,
            method="wilcoxon",
            n_genes=adata.n_vars,       # 取所有基因，后续按阈值筛选
            key_added=_key,
            use_raw=False,
        )
        _df = sc.get.rank_genes_groups_df(adata, group=_num, key=_key)
        _df["contrast"] = f"{_num}_vs_{_den}"
        _disease_deg_results[(_num, _den)] = _df

        _sig = _df[(_df["pvals_adj"] < PVAL_ADJ_CUTOFF) &
                    (abs(_df["logfoldchanges"]) > LOG2FC_CUTOFF)]
        _up = (_sig["logfoldchanges"] > 0).sum()
        _dn = (_sig["logfoldchanges"] < 0).sum()
        print(f"    显著 DEG: {len(_sig)} (上调={_up}, 下调={_dn})")

    print(f"\n跨疾病 DEG 完成: {len(_disease_deg_results)} 组对比")
else:
    print(f"DISEASE_COL '{DISEASE_COL}' 不在 obs 列中，跳过跨疾病 DEG")
    print(f"  可用 obs 列: {list(adata.obs.columns)}")

## 可视化

### 火山图：展示单个对比的显著性 vs 效应量

每个点为一个基因，横轴=log2 fold change，纵轴=-log10(调整后 p 值)。
红色=显著 DEG（超过 logFC 和 p 值双阈值）。
**为什么画火山图？** 它同时展示统计显著性和生物学效应量，
帮助 PI 快速识别"高 fold change + 强显著性"的明星基因。

In [ ]:
# 为每组疾病对比画火山图。
if _disease_col_available and _disease_deg_results:
    _n_cols = min(2, len(_disease_deg_results))
    _n_rows = (len(_disease_deg_results) + _n_cols - 1) // _n_cols
    fig, axes = plt.subplots(_n_rows, _n_cols, figsize=(6 * _n_cols, 5 * _n_rows))
    if _n_rows * _n_cols == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for _ax, ((_num, _den), _de_df) in zip(axes, _disease_deg_results.items()):
        _de_df["-log10_padj"] = -np.log10(_de_df["pvals_adj"].clip(lower=1e-300))
        _de_df["significant"] = (
            (_de_df["pvals_adj"] < PVAL_ADJ_CUTOFF) &
            (abs(_de_df["logfoldchanges"]) > LOG2FC_CUTOFF)
        )

        # 标注 top 10 基因
        _top = _de_df[_de_df["significant"]].nsmallest(10, "pvals_adj")

        _ax.scatter(
            _de_df[~_de_df["significant"]]["logfoldchanges"],
            _de_df[~_de_df["significant"]]["-log10_padj"],
            s=3, color="grey", alpha=0.3, rasterized=True,
        )
        _ax.scatter(
            _de_df[_de_df["significant"]]["logfoldchanges"],
            _de_df[_de_df["significant"]]["-log10_padj"],
            s=6, color="red", alpha=0.7, rasterized=True,
        )
        # 标注 top 基因名
        for _, _row in _top.iterrows():
            _ax.annotate(
                _row["names"],
                (_row["logfoldchanges"], _row["-log10_padj"]),
                fontsize=6, alpha=0.9,
            )

        _ax.axhline(-np.log10(PVAL_ADJ_CUTOFF), ls="--", color="gray", lw=0.8)
        _ax.axvline(LOG2FC_CUTOFF, ls="--", color="gray", lw=0.8)
        _ax.axvline(-LOG2FC_CUTOFF, ls="--", color="gray", lw=0.8)
        # 图表标题使用中文方便识别，轴标签保留英文遵循领域惯例
        _ax.set_title(f"差异表达：{_num} vs {_den}")
        _ax.set_xlabel("log2 Fold Change")
        _ax.set_ylabel("-log10(p_adj)")

    # 隐藏多余 subplot
    for _ax in axes[len(_disease_deg_results):]:
        _ax.set_visible(False)

    plt.tight_layout()
    _volcano_path = "results/figures/07_deg_volcano.png"
    fig.savefig(_volcano_path, dpi=200, bbox_inches="tight")
    plt.close("all")
    print(f"火山图已保存: {_volcano_path}")
else:
    print("无疾病对比结果，跳过火山图")


## 结果保存

- Per-cluster 标记基因完整表 → `results/tables/07_per_cluster_markers.csv`
- 每组疾病对比显著 DEG → `results/tables/07_disease_deg_{num}_vs_{den}.csv`
- 运行元数据 → `adata.uns["07_deg_v1"]`
- 产出 h5ad → `OUTPUT_PATH`

In [ ]:
# 导出每组疾病对比的 DEG 表并写入 adata.uns。
import datetime as _dt

_07_uns = {
    "method": "wilcoxon rank_genes_groups",
    "leiden_col": LEIDEN_COL,
    "disease_col": DISEASE_COL if _disease_col_available else None,
    "contrasts": [],
    "per_cluster_markers_csv": "results/tables/07_per_cluster_markers.csv",
    "timestamp": _dt.datetime.now().isoformat(),
}

# 导出疾病 DEG
for (_num, _den), _de_df in _disease_deg_results.items():
    _csv = f"results/tables/07_disease_deg_{_num}_vs_{_den}.csv"
    _de_df.to_csv(_csv, index=False)
    print(f"  疾病 DEG 已保存: {_csv} ({len(_de_df)} 基因)")
    _07_uns["contrasts"].append({
        "numerator": _num,
        "denominator": _den,
        "csv": _csv,
        "n_sig": int(((_de_df["pvals_adj"] < PVAL_ADJ_CUTOFF) &
                       (abs(_de_df["logfoldchanges"]) > LOG2FC_CUTOFF)).sum()),
    })

# 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 命名一致）
adata.uns["stage"] = "07_deg"     # 本 stage 标识
adata.uns["version"] = OUTPUT_VERSION                  # 与 OUTPUT_PATH 版本号一致
adata.uns["upstream"] = [UPSTREAM_PATH]
adata.uns["status"] = "experimental"  # PI 审查后改为 "promoted"


adata.uns[f"07_deg_{OUTPUT_VERSION}"] = _07_uns
print("运行元数据已写入 adata.uns['07_deg_{OUTPUT_VERSION}']")

In [ ]:
# 内存自检——确保 X 没有被误转为 dense。
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量被破坏: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32")

In [ ]:
# 写出 checkpoint。
adata.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"已写出 {OUTPUT_PATH}")
assert os.path.exists(OUTPUT_PATH), f"输出不存在: {OUTPUT_PATH}"
print(f"已验证: {OUTPUT_PATH} ({os.path.getsize(OUTPUT_PATH):,} bytes)")

In [ ]:
# 释放内存。
del adata
gc.collect()
print("内存已释放")